In [27]:
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_community  .document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rich import print

In [28]:
from dotenv import load_dotenv

load_dotenv()


True

In [29]:
model_name = "gpt-3.5-turbo"

In [30]:
# Carregar Modelos (Embeddings e LLM)

embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(
  model=model_name,
  temperature=0.7,
  max_tokens=200,
)

In [31]:
# Carregar PDF

pdf_link = "../documents/DOC-MPV-13082025-20250808.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split() # Carrega e divide as páginas do PDF

In [32]:
# Separamento em chunks (pedaços menores do documento)

text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=4000, # Tamanho do chunk
  chunk_overlap=200, # Overlap entre os chunks
  length_function=len, # Função que define o tamanho do chunk
  is_separator_regex=False, # Se o separador é uma regex
  add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [33]:
# Salvar no Vector Store (ChromaDB)

db = Chroma.from_documents(chunks, embedding=embeddings_model, persist_directory="../databases/chroma_db")

In [34]:
# Carregar DB
vectordb = Chroma(persist_directory="../databases/chroma_db", embedding_function=embeddings_model) 

# Load Retriever (Busca semelhante aos top 3 documentos relevantes)
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Criar uma chain para o LLM responder perguntas
chain = load_qa_chain(llm, chain_type="stuff")

In [35]:
def ask(question: str) -> str:
    """Faz uma pergunta e retorna a resposta"""
    context = retriever.invoke(question)
    answer = chain.invoke({
      "input_documents": context,
      "question": question
    })['output_text']

    return answer

In [36]:
user_question = input("Faça sua pergunta: ")

answer_llm = ask(user_question)

response = f"""Resposta do modelo {model_name}:

{answer_llm}"""

print(response)

Resposta do modelo gpt-3.5-turbo:

Esse arquivo foi gerado no ano de 2025.